# PrismRunner Interactive Walkthrough

This notebook lets you interactively test all core features of `prismrunner`:
1. **Hardware, Dtype & Context-Budget Auto-Detection**
2. **Dynamic Pydantic Schema-to-Prompt Compilation**
3. **Pre-Validation JSON Sanitization & Output Parsing (`<think>` stripping & truncated JSON salvage)**
4. **Batch DataFrame Extraction (`input_data=df`) & Expanding Results into Columns**
5. **Looping Through a DataFrame (`df.iterrows()`) for Row-by-Row Extraction**
6. **Token-Aware Sliding-Window Chunking & Deterministic Stitching**

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path (or run: %pip install -e /path/to/PrismRunner)
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "usertests" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import json
import pandas as pd
from typing import Literal, Optional, List
from pydantic import BaseModel, Field

from prismrunner import (
    DEFAULT_MODEL_NAME,
    PrismPipeline,
    Record,
    TokenAwareChunker,
    DeterministicStitcher,
    UniversalOutputParser,
    compile_schema_to_prompt,
    generate_hardware_advisory,
    get_optimal_device,
    resolve_device_and_dtype,
)

## 1. Hardware & Context Budget Inspection

In [ ]:
device_name, dtype = resolve_device_and_dtype()
device = get_optimal_device()
advisory = generate_hardware_advisory(DEFAULT_MODEL_NAME, context_limit=32768, device=device)

print(f"Active Device : {device_name} | Optimal Dtype: {dtype}")
print(json.dumps(advisory, indent=2))

## 2. Define a Custom Pydantic Schema & Compile to Prompt

In [ ]:
class ClinicalFinding(BaseModel):
    condition: str = Field(description="Diagnosed medical condition or symptom")
    status: Literal["active", "resolved", "suspected"] = Field(description="Clinical status")
    severity: int = Field(ge=1, le=5, description="Severity rating from 1 (mild) to 5 (critical)")
    notes: Optional[str] = Field(default=None, description="Additional clinical context")

compiled = compile_schema_to_prompt(ClinicalFinding)
print(compiled.render_prompt_block(as_array=True))

## 3. Test the Output Parser on Messy LLM Generations
Demonstrates `<think>` block stripping, markdown fence removal, trailing comma repair, phantom null filtering, and truncated JSON array recovery.

In [ ]:
raw_llm_response = """<think>
Let's check array[0] in the patient chart first.
</think>
```json
[
  {"condition": null, "status": null, "severity": null, "notes": null},
  {"condition": "Type 2 Diabetes", "status": "active", "severity": 3, "notes": "A1c at 7.8%",}
]
```
"""

parse_res = UniversalOutputParser.parse(raw_llm_response, model_cls=ClinicalFinding)
print("Success:", parse_res.success)
print("Validated Items:", [item.model_dump() for item in parse_res.items])

## 4. Batch Extraction Across a Pandas DataFrame (`input_data=df`)
Pass the entire DataFrame directly to `pipeline.run()` for length-bucketed batch execution, or loop over the returned records to flatten the extracted items into a relational DataFrame.

Set `LIVE_INFERENCE = True` to load the local LLM weights and run live generation.

In [ ]:
LIVE_INFERENCE = False  # Change to True to run live local LLM generation

df = pd.DataFrame({
    "note_id": ["N-01", "N-02", "N-03"],
    "department": ["Neurology", "Pulmonology", "Cardiology"],
    "text": [
        "Patient presents with acute migraine (severity 4/5). Prior seasonal allergies have resolved.",
        "Suspected mild sleep apnea (2/5); referred for overnight polysomnography.",
        "Active hypertension (3/5) controlled on lisinopril; chest pain ruled out (resolved, 1/5).",
    ],
})

pipeline = PrismPipeline(model_name=DEFAULT_MODEL_NAME)
batch_results = pipeline.run(
    task="extract",
    input_data=df,
    text_column="text",
    id_column="note_id",
    schema_cls=ClinicalFinding,
    dry_run=not LIVE_INFERENCE,
)

print(f"Processed {len(batch_results)} records in batch mode:")
print(json.dumps(batch_results[0], indent=2, default=str))

## 5. Looping Through a DataFrame (`df.iterrows()`) for Row-by-Row Extraction
If you want to iterate through a DataFrame row-by-row (e.g., to customize prompts per row, log progress, or append extracted Pydantic models into a flat table as you go), initialize `PrismPipeline` **once** outside the loop so the model stays warm in GPU/MPS memory across iterations.

In [ ]:
extracted_rows = []

# Simulated LLM JSON responses per row when LIVE_INFERENCE=False
mock_row_outputs = {
    "N-01": '[{"condition": "Acute Migraine", "status": "active", "severity": 4, "notes": "Presenting complaint"}, {"condition": "Seasonal Allergies", "status": "resolved", "severity": 1, "notes": "Prior history"}]',
    "N-02": '[{"condition": "Sleep Apnea", "status": "suspected", "severity": 2, "notes": "Referred for polysomnography"}]',
    "N-03": '[{"condition": "Hypertension", "status": "active", "severity": 3, "notes": "Controlled on lisinopril"}]',
}

for idx, row in df.iterrows():
    note_id = row["note_id"]
    dept = row["department"]
    clinical_text = row["text"]

    if LIVE_INFERENCE:
        # Runs live inference on this row using the already-loaded pipeline engine
        row_result = pipeline.run(
            task="extract",
            input_data=clinical_text,
            schema_cls=ClinicalFinding,
            instruction=f"Extract clinical findings relevant to {dept}.",
        )[0]
        findings = row_result.get("extracted_items", [])
    else:
        # Dry-run preview + simulated parse so you can test the DataFrame loop without downloading model weights
        _dry_payload = pipeline.run(
            task="extract",
            input_data=clinical_text,
            schema_cls=ClinicalFinding,
            instruction=f"Extract clinical findings relevant to {dept}.",
            dry_run=True,
        )[0]
        parsed = UniversalOutputParser.parse(mock_row_outputs[note_id], model_cls=ClinicalFinding)
        findings = [item.model_dump() for item in parsed.items]

    print(f"Row {idx} ({note_id} - {dept}): extracted {len(findings)} finding(s)")
    for finding in findings:
        extracted_rows.append({
            "note_id": note_id,
            "department": dept,
            **finding,
        })

# Convert the collected extractions into a clean flat DataFrame
extracted_df = pd.DataFrame(extracted_rows)
display(extracted_df)

## 6. Sliding-Window Chunking & Multi-Chunk Deduplication

In [ ]:
long_text = (
    "Patient history includes active hypertension managed with Lisinopril. "
    "Follow-up examination confirmed hypertension remains active alongside resolved bronchitis. "
    "New onset suspected osteoarthritis noted in the right knee."
)

chunker = TokenAwareChunker(tokenizer=None, chunk_size=18, chunk_overlap=4, strategy="sentence")
chunks = chunker.chunk_record(Record(id="doc_long", index=0, text=long_text))
for c in chunks:
    print(f"Chunk {c.chunk_index + 1}/{c.total_chunks}: {c.text}")